# A-SCLC

Edit the input path and device parameters, load the measurements, and inspect
the measured current density.
CSV format: `U(V),I(A)` with two numeric columns in that order (volts,
amperes). Raw measurements retain their signs and order.

In [1]:
from pathlib import Path
import sys

import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from asclc_workflow import (run_calculation, run_effective_mobility,
                            run_carrier_densities, run_model_occupations,
                            run_model_fermi_level, run_model_current, run_model_mobility, export_results)
from asclc_backend import equal_density_voltage, trap_dos, K_B, E_CHARGE
from asclc_plotting import (plot_measured_jv, plot_smoothed_jv,
                            plot_local_gamma, plot_effective_mobility,
                            plot_carrier_densities, plot_carrier_fraction,
                            plot_dos, plot_model_occupations, plot_model_fermi_level,
                            plot_model_current, plot_model_mobility)

## Input and sample parameters

Sample: MAPbBr3, S2, dark. Replace for your sample.


In [2]:
input_path = project_root / "data/MAPbBr3_S2_dark.csv"


experiment = dict(
    date="27.09.2024",
    sample="S2",
    regime="dark",
    voltage=(3, -3),   # V
    T=299.00,          # K
    T0=273.15,         # K
)


material = dict(
    L=6.00e-4,      # m, thickness
    S=7.70e-6,      # m^2, area
    eps_r=25.50,    # relative permittivity
    E_c=-3.36,      # eV, conduction band edge
    E_v=-5.58,      # eV, valence band edge
    E_g=2.22,       # eV, band gap
    m_eff_p=0.305,  # hole effective mass / m_e
    m_eff_e=0.320,  # electron effective mass / m_e
)

# Model: hand-selected parameters, Table S2 "Variable"
# (examples/MAPbBr3_S2.params). mu_0 is the reference line on the
# effective-mobility plot and the microscopic mobility of equation (5).
params = dict(
    mu_0=2.7e-3,    # m^2 V^-1 s^-1, microscopic mobility
    N_t=4.7e16,
    E_t=-4.82,
    T_t=30.0,
    E_F0=-4.84,
)




# docs/ASCLC_guide.md, "Local-slope smoothing window".
numerics = dict(window=11)


In [3]:
device = dict(area=material["S"], voltage_offset=0.0)
result = run_calculation(input_path, device=device)
print(f"Loaded {result.V.size} measurements; "
      f"{(~result.measurements.finite).sum()} nonfinite rows retained.")

Loaded 214 measurements; 0 nonfinite rows retained.


## Measured current density

Measured current density against voltage on logarithmic axes. Nonpositive
readings are kept in the data but cannot appear on a logarithmic axis.

In [4]:
figures = {}
fig, ax = plot_measured_jv(result)
figures["measured_jv"] = fig

## Smoothed current density

A centred `numerics["window"]`-point mean of the current density, drawn over the
raw markers at the same voltages so the two can be compared directly. Points
near either end, where the window would run off the data, have no smoothed
value.

In [5]:
fig_s, ax_s = plot_smoothed_jv(result, window=numerics["window"])
figures["smoothed_jv"] = fig_s

## Slope parameter and effective mobility

Guide steps A2 and A3. Each row covers the same trailing
`numerics["window"]`-point window starting at that row:

- `U` and `j`: trailing means of voltage and current density.
- `gamma`: Least-squares fit of `ln|U|` against `ln|j|` over the window,
  fitted on the raw rows.
- `mu_eff`: equation (4) of the paper on that row's `U`, `j` and `gamma`,

$$\mu_\mathrm{eff} = \mu_0\Theta =
  \frac{L^3}{\varepsilon_0\varepsilon_r(1-\gamma)(2-\gamma)^2}\,\frac{j}{U^2}.$$

The anchor is trailing, so a row describes the window just above it and the last
`window - 1` rows have no value. Plot against `mobility.U`, the voltage each row
describes.

In [6]:
mobility = run_effective_mobility(result, material=material,
                                  window=numerics["window"])
fig_g, ax_g = plot_local_gamma(result, mobility)
figures["local_gamma"] = fig_g
print(f"gamma, first six rows: "
      f"{np.array2string(mobility.gamma[:6], precision=4)}")

gamma, first six rows: [-0.1449  0.1648 -0.0097 -0.0201 -0.0279  0.0532]


Equation (4) diverges at `gamma = 1` (ohmic) and returns a negative mobility
where `gamma > 1`; such points stay in the data but cannot appear on a
logarithmic axis. The first decade of bias is at the instrument's noise floor,
where `gamma` is fitted through sign-flipping picoamps — read the physics above
about 0.3 V.

In [7]:
fig_m, ax_m = plot_effective_mobility(result, mobility, mu_0=params["mu_0"])
figures["effective_mobility"] = fig_m
finite = np.isfinite(mobility.mu_eff)
print(f"mu_eff defined at {finite.sum()} of {finite.size} points; "
      f"maximum {np.nanmax(mobility.mu_eff):.3e} m^2 V^-1 s^-1 "
      f"({np.nanmax(mobility.mu_eff) / params['mu_0']:.2f} x mu_0)")

mu_eff defined at 204 of 214 points; maximum 5.998e-04 m^2 V^-1 s^-1 (0.22 x mu_0)


## Free and trapped charge carriers

Guide step A4, on the same rows as step A3. Equation (5) is Ohm's law for these
devices solved for the free concentration, and equation (6) is what is left when
equation (2) is divided by it:

$$p_\mathrm{f} = \frac{L\,j}{e\mu_0(2-\gamma)U},\qquad
  \frac{p_\mathrm{f}}{\Theta} =
  \frac{\varepsilon_0\varepsilon_r(1-\gamma)(2-\gamma)U}{eL^2}.$$

The article labels the second quantity $p_\mathrm{t}$ and then sets
$\Theta = p_\mathrm{f}/(p_\mathrm{f}+p_\mathrm{t})$, which cannot hold together
with equations (2) and (4). Read as the *total* concentration $p_\mathrm{s}$,
with $p_\mathrm{t} = p_\mathrm{s} - p_\mathrm{f}$, the equations become one
system with a single $\Theta = \mu_\mathrm{eff}/\mu_0$ — equal to 1 in the
Mott–Gurney limit and to 1/2 where $p_\mathrm{t} = p_\mathrm{f}$, as the article
states for both regions. That is `theta_model="free_over_total"`, the default;
`"absolute_over_total"` (the SI read literally) and `"mobility_ratio"` (the mobility ratio) are the alternatives. Only `mu_0` separates the three
$p_\mathrm{f}$ curves from each other — it scales $p_\mathrm{f}$ and leaves
equation (6) untouched.

In [8]:
carriers = run_carrier_densities(mobility, material=material,
                                 mu_0=params["mu_0"])
fig_p, ax_p = plot_carrier_densities(carriers)
figures["carrier_densities"] = fig_p
i = np.nanargmax(carriers.U)
print(f"at U = {carriers.U[i]:.2f} V:  p_f = {carriers.p_f[i]:.3e},  "
      f"p_t = {carriers.p_t[i]:.3e},  p_s = {carriers.p_s[i]:.3e} m^-3")
print(f"largest p_t: {np.nanmax(carriers.p_t):.3e} m^-3 "
      f"({np.nanmax(carriers.p_t) / 1e6:.3e} cm^-3)")

at U = 2.93 V:  p_f = 3.667e+15,  p_t = 1.885e+16,  p_s = 2.252e+16 m^-3
largest p_t: 1.952e+16 m^-3 (1.952e+10 cm^-3)


### Parameter theta

$\Theta = p_\mathrm{f}/p_\mathrm{s}$, the fraction of the injected charge that
is free. Where it reaches 1/2 the trapped and free concentrations are equal, the
charge starts to occupy the transport band and the Mott–Gurney law applies;
`equal_density_voltage` returns the voltages at which that happens. An empty
result means the scan never gets there for the `mu_0` in use.

In [9]:
fig_t, ax_t = plot_carrier_fraction(carriers)
figures["carrier_fraction"] = fig_t
crossings = equal_density_voltage(carriers.U, carriers.p_f, carriers.p_t)
peak = np.nanargmax(carriers.theta)
print(f"Theta peaks at {carriers.theta[peak]:.3f} "
      f"(U = {carriers.U[peak]:.2f} V, mu_eff = {mobility.mu_eff[peak]:.3e})")
print(f"p_t = p_f at: {np.array2string(crossings, precision=3)} V"
      if crossings.size else "p_t = p_f nowhere in this scan: Theta stays "
      f"below 1/2, so the charge never fills the transport band at "
      f"mu_0 = {params['mu_0']:.1e} m^2 V^-1 s^-1")

Theta peaks at 0.222 (U = 2.54 V, mu_eff = 5.998e-04)
p_t = p_f nowhere in this scan: Theta stays below 1/2, so the charge never fills the transport band at mu_0 = 2.7e-03 m^2 V^-1 s^-1


## Density of states

The two transport bands are parabolic. The localized trap profile is

$$g_t(E) = \frac{N_t}{4 k_B T_t \cosh[(E-E_t)/(k_B T_t)]}.$$

It integrates to $(\pi/4)N_t$, so $N_t$ is a concentration scale.
The next step evaluates occupations of these states.


In [10]:
dos = dict(E_v=material["E_v"], E_c=material["E_c"],
           m_eff_h=material["m_eff_p"], m_eff_e=material["m_eff_e"],
           N_t=params["N_t"], E_t=params["E_t"], T_t=params["T_t"])

fig_d, ax_d = plot_dos(
    np.linspace(material["E_v"] - .8, material["E_c"] + .7, 36001), **dos)
figures["dos"] = fig_d

# The same function on a grid fine enough to resolve the trap: k_B T_t is

# samples this whole profile in about five points.
kt_t = K_B*params["T_t"]/E_CHARGE
close_up = np.linspace(params["E_t"] - 27*kt_t, params["E_t"] + 27*kt_t, 20001)
peak = params["N_t"]/(4*kt_t)
fig_dt, ax_dt = plot_dos(close_up, **dos)
ax_dt.set_yscale("linear")
ax_dt.set_ylim(0, 1.1*peak)
figures["dos_trap"] = fig_dt

saturation = np.trapezoid(
    trap_dos(close_up, N_t=params["N_t"], E_t=params["E_t"], T_t=params["T_t"]),
    close_up)
print(f"k_B T_t = {kt_t*1e3:.2f} meV; trap peaks at {peak:.3e} m^-3 eV^-1")
print(f"trap integral {saturation:.4e} m^-3 = {saturation/params['N_t']:.4f} N_t "
      f"(pi/4 = {np.pi/4:.4f})")

k_B T_t = 2.59 meV; trap peaks at 4.545e+18 m^-3 eV^-1
trap integral 3.6914e+16 m^-3 = 0.7854 N_t (pi/4 = 0.7854)


## M2: Model occupations

Free populations are absolute, total populations are equilibrium-subtracted,
and the labelled trapped populations subtract the equilibrium free value.
Theta is `abs(free / total)` and can exceed one or diverge. Signed values
remain in `model`; nonpositive values leave gaps on logarithmic plots.


In [ ]:

model_energy = -np.arange(3001) * 0.003
model_fermi = -9.0 + np.arange(3001) * 0.003
model = run_model_occupations(model_energy, model_fermi, material=material,
                              params=params, temperature=experiment["T"])
fig_m2, axes_m2 = plot_model_occupations(model)
axes_m2.set_ylim(bottom=10e10)
axes_m2.set_xlim(-2, -7)
figures["model_occupations"] = fig_m2
print(f"At EF0 = {model.E_F0:g} eV: nf0 = {model.n_f0:.6e}, "
      f"pf0 = {model.p_f0:.6e} m^-3")


## M3: Model Fermi level shift

At the temperature used in M2, each occupation row already has a quasi-Fermi
level $E_F$. Carry that coordinate forward and calculate the signed hole
separation $\Delta E_F=E_F-E_v$. This is distinct from $E_F-E_{F0}$.
In the nondegenerate limit, equation (7) gives
$\Delta E_F=k_BT\ln(N_v/p_f)$; the occupation sweep also covers states
outside that limit, so its energy coordinate is retained directly.

The plot pairs this separation with the absolute free-hole density from M2.
Negative separations denote levels inside the valence band.


In [ ]:
model_levels = run_model_fermi_level(model, material=material)
fig_m3, ax_m3 = plot_model_fermi_level(model, model_levels)
figures["model_fermi_level"] = fig_m3
print(f"M2 temperature: {experiment['T']:g} K")
print(f"EF: {model_levels.E_F.min():.3f} to {model_levels.E_F.max():.3f} eV")
print(f"EF - Ev: {model_levels.delta_E_v.min():.3f} to "
      f"{model_levels.delta_E_v.max():.3f} eV")
print(f"EF0 - Ev: {model_levels.E_F0 - material['E_v']:.3f} eV")


## M4: Model current density

On the existing energy sweep, use $q=n_t$ or $p_t$ and the corresponding
absolute free density $f$:

$$U=\frac{e L^2 q}{\varepsilon_0\varepsilon_r(1-\gamma)(2-\gamma)},
\qquad J=\frac{e\mu_0 f(2-\gamma)U}{L}.$$

Retain $\gamma=T_t/T$. The density mapping $q=s-f_0$ is an algebraic
continuation: it is not generally the total space charge required by
Poisson's equation. Electrons and holes are separate single-carrier curves.
Signed results remain in the arrays; nonpositive points leave gaps on the
logarithmic plot. The supplied gamma is a shape coefficient and need not
equal the slope measured along the resulting curve.


In [ ]:
model_gamma = params["T_t"] / experiment["T"]
model_current = run_model_current(model, material=material,
                                  mu_0=params["mu_0"], gamma=model_gamma)
fig_m4, ax_m4 = plot_model_current(model_current, measured=result)
figures["model_current"] = fig_m4
print(f"M4 gamma: {model_current.gamma:.8g}")


## M5: Model effective mobility

For each carrier, $\mu_\mathrm{eff}=\mu_0\Theta$ uses the M2 ratio and the
existing microscopic mobility. No parameter optimization is performed.
The M2 ratios mix absolute free and excess total populations, so values
above one and singularities remain in the result; this is not always a
bounded physical free-carrier fraction. Nonpositive or nonfinite mobilities
leave gaps on the logarithmic plot. Both carriers use the supplied `mu_0`.


In [ ]:
model_mobility = run_model_mobility(model, mu_0=params["mu_0"])
fig_m5, ax_m5 = plot_model_mobility(model_mobility)
figures["model_mobility"] = fig_m5
print(f"Microscopic mobility: {model_mobility.mu_0:g} m^2 V^-1 s^-1")


## Export

Save the figure as PNG/PDF/SVG and write the measured CSV to the output
directory.

In [11]:
output_dir = export_results(result, figures, project_root / "outputs")
print(f"Saved {len(figures)} figures and measured data to {output_dir}")

Saved 8 figures and measured data to /home/vk/WORK/asclc/outputs
